In [1]:
## login wandb
import wandb
wandb.login()
## set up project name
import os
os.environ["WANDB_PROJECT"] = "chess-llm" 
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

wandb: Currently logged in as: norrawee to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
import unsloth
import vllm
import torch
import trl

print(vllm.__version__)
print(unsloth.__version__)
print(torch.__version__)
print(trl.__version__)

/home/azureuser/earth/global-chess-challenge-2025-experiments/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 01-16 02:05:30 [__init__.py:216] Automatically detected platform cuda.


W0116 02:05:31.936000 3685 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0116 02:05:31.936000 3685 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


🦥 Unsloth Zoo will now patch everything to make training faster!
0.10.2
2026.1.2
2.8.0+cu128
0.24.0


## Model

In [4]:
from unsloth import FastLanguageModel

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Norrawee/Qwen3-4B-Thinking-2507-exp04", 
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = False, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

==((====))==  Unsloth 2026.1.2: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [5]:
NEW_TOKENS = [  
    "♔","♕","♖","♗","♘","♙",  
    "♚","♛","♜","♝","♞","♟",
    "<uci_move>", "</uci_move>",
]
xs = tokenizer("♔♕♖♗♘♙♚♛♜♝♞♟ <uci_move>a1a2</uci_move>")
print([tokenizer.decode(x) for x in xs["input_ids"]])

['♔', '♕', '♖', '♗', '♘', '♙', '♚', '♛', '♜', '♝', '♞', '♟', ' <', 'uci', '_move', '>a', '1', 'a', '2', '</', 'uci', '_move', '>']


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

Unsloth 2026.1.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## Data

In [7]:
from datasets import load_dataset, Dataset
from tqdm import tqdm
import chess
import pandas as pd

In [8]:
SYSTEM_PROMPT = """You are an expert chess player.
Given the position, choose the best move in uci format enclosed by <uci_move> </uci_move>.

Position: {pad_fen}
Moves: {legal_moves_uci_list}
Your turn: {side_to_move}

Think 100 words maximum"""

In [9]:
from datasets import load_dataset  
  
dataset = load_dataset("Norrawee/sft-exp05")
df = dataset["train"].to_pandas()
df = df[:100]

README.md:   0%|          | 0.00/481 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/29.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/115846 [00:00<?, ? examples/s]

In [15]:
def pad_fen(fen):  
    parts = fen.split(" ")  
    board = parts[0]  
  
    padded_rows = []  
    for row in board.split("/"):  
        expanded = ""  
        for char in row:  
            if char.isdigit():  
                expanded += "•" * int(char)  # pad empty squares  
            else:  
                expanded += char  
        padded_rows.append(expanded)  
  
    parts[0] = "/".join(padded_rows)  
    return " ".join(parts)  

def format_prompt(row):  
    prompt = SYSTEM_PROMPT.format(
        side_to_move=row["side_to_move"],
        legal_moves_uci_list=" ".join(row["legal_moves_uci_list"]),
        pad_fen=pad_fen(row["FEN"]),
    ) 
    response = f"<think>\n{row["explanation"]}\n</think>\n<uci_move>{row["target_move"]}</uci_move>"   
    return [  
        {"role": "user", "content": prompt},  
        {"role": "assistant", "content": response},  
    ]

In [16]:
## preprocess
df["prompt"] = df.apply(format_prompt, axis=1)
df["text"] = tokenizer.apply_chat_template(df["prompt"].values.tolist(), tokenize=False)

In [17]:
from sklearn.model_selection import train_test_split  
from datasets import Dataset  
  
# Unique boards  
unique_boards = df["FEN"].unique()  
  
# Split boards, NOT rows  
train_boards, test_boards = train_test_split(  
    unique_boards,  
    test_size=0.1 if len(df) < 500 else 50,  
    random_state=42,  
    shuffle=True,  
)  
  
# Filter rows  
train_df = df[df["FEN"].isin(train_boards)].reset_index(drop=True)  
test_df  = df[df["FEN"].isin(test_boards)].reset_index(drop=True)  
  
# Create HF datasets  
ds = {  
    "train": Dataset.from_pandas(train_df),  
    "test": Dataset.from_pandas(test_df),  
}  

In [18]:
text = ds["train"]["text"][0]

print(text)
print(len(tokenizer(text)["input_ids"]))

<|im_start|>user
You are an expert chess player.
Given the position, choose the best move in uci format enclosed by <uci_move> </uci_move>.

Position: rnbqkbnr/pppppppp/••••••••/••••••••/••••P•••/••••••••/PPPP•PPP/RNBQKBNR b KQkq - 0 1
Moves: g8h6 g8f6 b8c6 b8a6 h7h6 g7g6 f7f6 e7e6 d7d6 c7c6 b7b6 a7a6 h7h5 g7g5 f7f5 e7e5 d7d5 c7c5 b7b5 a7a5
Your turn: Black

Think 100 words maximum<|im_end|>
<|im_start|>assistant
<think>
1. Analyze board: Black's pawn on e7 controls center; White has a pawn on e5, but e7e5 challenges it directly.  
2. Move 1: e7e5 pushes pawn to central square, controls d4 and d5, supports future development and challenges White’s center.  
3. Move 2: e7e6 weakens the e-pawn, blocks the d8-bishop, and delays central control, allowing White to play e5e4.  
4. Conclusion: e7e5 is better; it controls center and develops quickly.
</think>

<uci_move>e7e5</uci_move><|im_end|>

342


## SFT

In [19]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset= ds["train"],
    eval_dataset= ds["test"],
    args = SFTConfig(
        dataset_text_field = "text",
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "wandb", # Use TrackIO/WandB etc
        
        # training params
        learning_rate=5e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        # num_train_epochs=5,
        fp16=False,
        bf16=True,
        weight_decay = 0.001,
        
        # logging
        # eval_strategy="epoch",
        # save_strategy="epoch",
        # logging_strategy="epoch",
        eval_strategy="steps",
        save_strategy="steps",
        logging_strategy="steps",
        logging_steps=10,
        save_steps=10,
        eval_steps=10,
        save_total_limit=1,
        max_steps=30,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=28):   0%|          | 0/90 [00:00<?, ? examples/s]

num_proc must be <= 10. Reducing num_proc to 10 for dataset of size 10.
[datasets.arrow_dataset|WARNING]num_proc must be <= 10. Reducing num_proc to 10 for dataset of size 10.


Unsloth: Tokenizing ["text"] (num_proc=10):   0%|          | 0/10 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [20]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 90 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,1.721100,1.329350
20,1.139900,1.005392
30,0.934100,0.920498


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▂▁
eval/runtime,█▁▁
eval/samples_per_second,▁██
eval/steps_per_second,▁██
train/epoch,▁▁▄▄███
train/global_step,▁▁▅▅███
train/grad_norm,█▂▁
train/learning_rate,█▅▁
train/loss,█▃▁
eval/loss,0.9205
eval/runtime,0.3342


TrainOutput(global_step=30, training_loss=1.2650184313456216, metrics={'train_runtime': 59.1819, 'train_samples_per_second': 8.111, 'train_steps_per_second': 0.507, 'total_flos': 4039282747852800.0, 'train_loss': 1.2650184313456216, 'epoch': 5.0})

## Test

In [21]:
import re
UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
  
def extract_uci(text):  
    match = UCI_PATTERN.search(text)  
    return match.group(1).strip() if match else None  

In [23]:
import re
UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
def extract_uci(text):  
    match = UCI_PATTERN.search(text)  
    return match.group(1).strip() if match else None  
 
import chess  
import chess.engine  
# ---------------- CONFIG ----------------  
ENGINE_PATH = "stockfish"   # change if needed  
ENGINE_LIMIT = chess.engine.Limit(time=1.0, depth=16) 

  
def evaluate(fen_board, uci_move, verbose=False):
    try:
        engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
        board = chess.Board(fen_board)  

        ## before 
        info_before = engine.analyse(board, ENGINE_LIMIT)  
        eval_before = info_before["score"].relative.score(mate_score=1000) 

        move = chess.Move.from_uci(uci_move) 
        board.push(move)  

        ## after
        info_after = engine.analyse(board, ENGINE_LIMIT)  
        eval_after = - info_after["score"].relative.score(mate_score=1000)
        
        engine.quit()
        
        if verbose:
            print(eval_before, eval_after)
        
        ## calculate reward
        delta = eval_after - eval_before
        if delta > 100:
            return 2
        elif delta > -100:
            return 1
        else:
            return 0
    except:
        print("Error")
        return -1
  


In [24]:
def generate_batch(model, prompts):  
    texts = [  
        tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)  
        for p in prompts  
    ]  
  
    model_inputs = tokenizer(  
        texts,  
        return_tensors="pt",  
        padding=True,  
    ).to(model.device)  
  
    generated = model.generate(  
        **model_inputs,  
        max_new_tokens=200,
        do_sample=True,
        temperature=0.01,
        top_p=0.8,
        top_k=20,
    )   
  
    decoded = tokenizer.batch_decode(  
        generated,  
        skip_special_tokens=True  
    )  
  
    return [t.split("assistant")[-1].strip() for t in decoded]  

In [25]:
from tqdm import tqdm  
  
BATCH_SIZE = 1 ## padding affects the outputs (i dont know how to fix).
  
responses = []
for batch_start in tqdm(range(0, len(ds["test"]), BATCH_SIZE)):  
    batch = ds["test"][batch_start: batch_start + BATCH_SIZE]  
  
    prompts = [ex[:1] for ex in batch["prompt"]]  
    responses.extend(generate_batch(model, prompts))

100%|██████████| 10/10 [00:57<00:00,  5.77s/it]


In [28]:
outputs = []

for i, (response, example) in enumerate(zip(responses, ds["test"])):
    pred_move = extract_uci(response)  
    legal_moves_uci_list = example["legal_moves_uci_list"]
    fen = example["FEN"]

    if pred_move not in legal_moves_uci_list:
        legal = False
        score = -1000
    else:
        legal = True
        score = evaluate(fen, pred_move)
    outputs.append({
        "score": score,
        "response": response,
        "legal": legal
    })
    
    # if i < 10:
    #     print(example["board_utf"])
    #     print(response)
    #     print(score)
    #     print(f"Target move: {example["target_move"]}")
    #     print("*"*60)


In [29]:
output_df = pd.DataFrame(outputs)
output_df[["legal"]].value_counts()

legal
True     10
Name: count, dtype: int64

In [30]:
output_df[["score"]].describe()

,score
count,10.000000
mean,0.100000
std,0.316228
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


## Save

In [ ]:
model.push_to_hub_merged(
    "Norrawee/Qwen/Qwen2.5-7B-Instruct-sft-exp04", 
    tokenizer,
    save_method = "merged_16bit", 
)

In [ ]:
wandb.finish()